# Interactive Project Diagrams

This notebook builds separate interactive time-series diagrams from `DataDiagrams/test.json` and `DataDiagrams/test_virtual.json`.

Usage:
1. Keep both JSON files in the `DataDiagrams` folder.
2. Run all cells.


In [5]:
import json
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display

try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError("plotly is required. Install with: pip install plotly") from exc

BASE_DIR = Path("DataDiagrams")
JSON_SPECS = [
    ("Main Project", BASE_DIR / "test.json", "standard"),
    ("Virtual Project", BASE_DIR / "test_virtual.json", "virtual"),
]

missing_paths = [path for _, path, _ in JSON_SPECS if not path.exists()]
if missing_paths:
    missing_list = ", ".join(str(path) for path in missing_paths)
    raise FileNotFoundError(f"Missing input file(s): {missing_list}")

COLOR_MAP = {
    "historyBeforeNow": "#1f77b4",
    "historyFromProjectStartDateToNow": "#1f77b4",
    "predictionUntilTargetCompletionDate": "#ff7f0e",
    "predictionBetweenPrioritizationAndTargetCompletionDate": "#ff7f0e",
    "predictionAfterTargetCompletionDateWith100SuccessRate": "#2ca02c",
    "predictionAfterTargetCompletionDateWith0SuccessRate": "#d62728",
}

DASH_MAP = {
    "historyBeforeNow": "solid",
    "historyFromProjectStartDateToNow": "solid",
    "predictionUntilTargetCompletionDate": "dash",
    "predictionBetweenPrioritizationAndTargetCompletionDate": "dash",
    "predictionAfterTargetCompletionDateWith100SuccessRate": "dot",
    "predictionAfterTargetCompletionDateWith0SuccessRate": "dashdot",
}

SECTION_ORDER = [
    "historyBeforeNow",
    "historyFromProjectStartDateToNow",
    "predictionUntilTargetCompletionDate",
    "predictionBetweenPrioritizationAndTargetCompletionDate",
    "predictionAfterTargetCompletionDateWith100SuccessRate",
    "predictionAfterTargetCompletionDateWith0SuccessRate",
]


def extract_series(frame: pd.DataFrame, section_name: str):
    if frame.empty:
        return pd.DataFrame(columns=["date", section_name])

    frame = frame[["date", "value"]].copy()
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.sort_values(["date"]).reset_index(drop=True)
    return frame.rename(columns={"value": section_name})


def detect_virtual_default_granularity(root: dict) -> str:
    candidate_sections = [
        "historyBeforeNow",
        "historyFromProjectStartDateToNow",
        "predictionUntilTargetCompletionDate",
        "predictionBetweenPrioritizationAndTargetCompletionDate",
        "predictionAfterTargetCompletionDateWith100SuccessRate",
        "predictionAfterTargetCompletionDateWith0SuccessRate",
    ]
    dates = []
    for section_name in candidate_sections:
        section = root.get(section_name)
        if not isinstance(section, dict):
            continue
        for row in section.get("data", []):
            date_str = row.get("date")
            if date_str:
                dates.append(pd.to_datetime(date_str))

    if len(dates) < 2:
        return "Monthly"

    unique_dates = sorted(set(dates))
    if len(unique_dates) < 2:
        return "Monthly"

    day_diffs = pd.Series(unique_dates).diff().dropna().dt.days
    if day_diffs.empty:
        return "Monthly"

    median_days = float(day_diffs.median())
    if median_days <= 2:
        return "Daily"
    if median_days <= 10:
        return "Weekly"
    return "Monthly"


def load_payload(label: str, json_path: Path, dataset_type: str) -> dict:
    payload = json.loads(json_path.read_text())
    root = payload["data"] if "data" in payload else payload

    project_start = pd.to_datetime(root["projectStartDate"])
    project_end = pd.to_datetime(root["projectEndDate"])
    all_dates = pd.DataFrame({"date": pd.date_range(project_start, project_end, freq="D")})

    frame = all_dates.copy()
    traces = []
    for section_name in SECTION_ORDER:
        section = root.get(section_name)
        if not isinstance(section, dict) or "data" not in section:
            continue
        section_frame = extract_series(pd.DataFrame(section["data"]), section_name)
        frame = frame.merge(section_frame, on="date", how="left")
        traces.append(
            {
                "name": section_name,
                "section": section_name,
                "color": COLOR_MAP.get(section_name, "#444444"),
                "dash": DASH_MAP.get(section_name, "solid"),
            }
        )

    markers = [(project_start, "#7f7f7f", "Project Start")]
    if "prioritizationDate" in root:
        markers.append((pd.to_datetime(root["prioritizationDate"]), "#8c564b", "Prioritization Date"))
    markers.append((pd.to_datetime(root["targetCompletionDate"]), "#111111", "Target Completion"))
    if "actualCompletionDate" in root:
        markers.append((pd.to_datetime(root["actualCompletionDate"]), "#bcbd22", "Actual Completion"))
    markers.append((project_end, "#7f7f7f", "Project End"))

    return {
        "name": json_path.name,
        "label": label,
        "dataset_type": dataset_type,
        "default_granularity": detect_virtual_default_granularity(root) if dataset_type == "virtual" else "Daily",
        "df": frame,
        "traces": traces,
        "markers": markers,
        "project_start": project_start,
        "project_end": project_end,
    }


datasets = [load_payload(label, path, dataset_type) for label, path, dataset_type in JSON_SPECS]
has_virtual_dataset = any(dataset["dataset_type"] == "virtual" for dataset in datasets)
virtual_default_granularity = next(
    (dataset["default_granularity"] for dataset in datasets if dataset["dataset_type"] == "virtual"),
    "Monthly",
)


In [6]:
global_start = min(dataset["project_start"] for dataset in datasets)
global_end = max(dataset["project_end"] for dataset in datasets)

view_min_picker = widgets.DatePicker(description="Min Date", value=global_start.date())
view_max_picker = widgets.DatePicker(description="Max Date", value=global_end.date())
scenario_dropdown = widgets.Dropdown(
    options=["100% Success", "0% Success", "Both"],
    value="Both",
    description="Scenario"
)
virtual_granularity_dropdown = widgets.Dropdown(
    options=["Daily", "Weekly", "Monthly"],
    value=virtual_default_granularity if virtual_default_granularity in {"Daily", "Weekly", "Monthly"} else "Monthly",
    description="Virtual View",
)
cumulative_toggle = widgets.Checkbox(value=False, description="Cumulative")
outs = [widgets.Output() for _ in datasets]


def series_for_plot(series: pd.Series, cumulative: bool, granularity: str) -> pd.Series:
    prepared = series.fillna(0)
    if cumulative:
        prepared = prepared.cumsum()

    if granularity == "Daily":
        return prepared

    resample_rule = "W" if granularity == "Weekly" else "ME"
    agg = "last" if cumulative else "sum"
    return prepared.resample(resample_rule).agg(agg)


def include_trace(section: str, scenario: str) -> bool:
    if section == "predictionAfterTargetCompletionDateWith100SuccessRate":
        return scenario in ("100% Success", "Both")
    if section == "predictionAfterTargetCompletionDateWith0SuccessRate":
        return scenario in ("0% Success", "Both")
    return True


def build_figure(
    dataset: dict,
    view_min: pd.Timestamp,
    view_max: pd.Timestamp,
    scenario: str,
    cumulative: bool,
    granularity: str,
):
    filtered = dataset["df"][
        (dataset["df"]["date"] >= view_min) & (dataset["df"]["date"] <= view_max)
    ].copy()
    filtered = filtered.set_index("date")

    fig = go.Figure()
    for trace in dataset["traces"]:
        if not include_trace(trace["section"], scenario):
            continue
        y_series = series_for_plot(filtered[trace["name"]], cumulative, granularity)
        fig.add_trace(
            go.Scatter(
                x=y_series.index,
                y=y_series.values,
                mode="lines",
                name=trace["name"],
                line={"width": 2, "color": trace["color"], "dash": trace["dash"]},
            )
        )

    for x_value, color, label in dataset["markers"]:
        if view_min <= x_value <= view_max:
            fig.add_vline(x=x_value, line_color=color, line_dash="dash", line_width=1.5)
            fig.add_annotation(
                x=x_value,
                y=1,
                yref="paper",
                yanchor="bottom",
                text=label,
                showarrow=False,
                font={"size": 11, "color": color},
            )

    y_title = f"{granularity} Cumulative Value" if cumulative else f"{granularity} Value"
    fig.update_layout(
        title=dataset["label"],
        xaxis_title="Date",
        yaxis_title=y_title,
        template="plotly_white",
        hovermode="x unified",
        height=680,
        legend={"orientation": "h", "y": 1.1, "x": 0},
    )
    fig.update_xaxes(rangeslider_visible=True)
    return fig


def draw(_=None):
    if view_min_picker.value is None or view_max_picker.value is None:
        for out in outs:
            with out:
                out.clear_output(wait=True)
                print("Please set Min Date and Max Date.")
        return

    view_min = pd.Timestamp(view_min_picker.value)
    view_max = pd.Timestamp(view_max_picker.value)
    if view_min > view_max:
        for out in outs:
            with out:
                out.clear_output(wait=True)
                print("Min Date must be <= Max Date.")
        return

    for dataset, out in zip(datasets, outs):
        granularity = "Daily"
        if dataset["dataset_type"] == "virtual":
            granularity = virtual_granularity_dropdown.value
        with out:
            out.clear_output(wait=True)
            display(HTML(f"<h3>{dataset['label']}</h3>"))
            build_figure(
                dataset,
                view_min=view_min,
                view_max=view_max,
                scenario=scenario_dropdown.value,
                cumulative=cumulative_toggle.value,
                granularity=granularity,
            ).show()


for widget in (view_min_picker, view_max_picker, scenario_dropdown, cumulative_toggle, virtual_granularity_dropdown):
    widget.observe(draw, names="value")

display(widgets.HBox([view_min_picker, view_max_picker]))
display(widgets.HBox([scenario_dropdown, cumulative_toggle]))
if has_virtual_dataset:
    display(widgets.HBox([virtual_granularity_dropdown]))
for out in outs:
    display(out)

draw()


Output()

Output()